In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch.optim import Adam



In [2]:
batch_size = 64
img_size = 28
patch_size = 7
num_channels = 1
num_patches = (img_size // patch_size) ** 2
num_heads = 1
embed_dim = 16
mlp_dim = 16
transformer_units = 1

In [3]:
transform = transforms.Compose(
    [transforms.ToTensor()])


In [4]:
trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                        download=True, transform=transform)
valset = torchvision.datasets.MNIST(root='./data', train=False,
                                        download=True, transform=transform)

In [5]:
image,label=trainset[0]
image
label

5

In [6]:
train_data = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True)
val_data = torch.utils.data.DataLoader(valset, batch_size=batch_size,
                                          shuffle=False)

In [7]:
images, labels = next(iter(val_data))
#ps: nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1) means chose 32 filters and recreates 32 picture out of that same picture ( not really a picture a feature map ) ( fonction hedhi produces 32 " taswira jdida " kol filter yaaml taswira w zize mtaa taswira tout depend ml kernel size w ml padding)
# Print the shape of the images tensor
print("Shape of images in a batch:", images.shape)
patch_embed = nn.Conv2d(num_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
embedded_data = patch_embed(images)
print("Shape of embedded data after conv2d :", embedded_data.shape)
embedded_data = embedded_data.flatten(2)
embedded_data = embedded_data.transpose(1,2)
print("Shape of embedded data:", embedded_data.shape)



Shape of images in a batch: torch.Size([64, 1, 28, 28])
Shape of embedded data after conv2d : torch.Size([64, 16, 4, 4])
Shape of embedded data: torch.Size([64, 16, 16])


In [8]:
class PatchEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = nn.Conv2d(num_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2)
        x = x.transpose(1,2)
        return x

In [9]:
class TransformerArchitecture(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(embed_dim) #Stabilizes the representation before the next operation.
        self.self_attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
# X
# │
# ├── Linear → Q
# ├── Linear → K
# └── Linear → V
    #    │
      #  ▼
   # Q × Kᵀ
     #   │
       # ▼
   # attention scores
     #   │
       # ▼
     # softmax
       # │
       # ▼
 # attention weights
   #     │
     #   ▼
 # weights × V
   #     │
     #   ▼
     #   split into
      # 12 heads
     #     ↓
   #attention independently
   #in each head
     #     ↓
      #concatenate
         # ↓
    #output projection
          #↓
   #attention_output
 # multi-head output : this happens in multiheadattention 
        # multiheadattention returns attention_output and attention_weights that s why we add [0]
        # 
        self.layer_norm_2 = nn.LayerNorm(embed_dim)
        self.multi_layer_perceptron = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim)
        )
    def forward(self, x):
        residual_1 = x
        attention_output = self.self_attention(self.layer_norm_1(x),self.layer_norm_1(x),self.layer_norm_1(x))[0] #3 inputs cause not all attention are self attention so we define that we need K,Q, and v
        x = attention_output + residual_1
        residual_2 = x
        mlp_output = self.multi_layer_perceptron(self.layer_norm_2(x))
        x = mlp_output + residual_2
        return x


In [10]:
class VisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embedding = PatchEmbedding()
        self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim)) # This tensor is learnable. cause nn.parametres , during training pytorch modifies these values using backporpagation it initially contains 16 random values ( 1, 1 ) - 1 copy 1 patch 
        self.pos_embed = nn.Parameter(torch.randn(1, (img_size // patch_size) ** 2 + 1, embed_dim)) #( 1 copy , 17 patches,16 dim )  
        self.transformer_layers = nn.Sequential(*[TransformerArchitecture() for _ in range(transformer_units)])

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 10)
        )

    def forward(self,x):
        x = self.patch_embedding(x)
        B = x.size(0) #(batch size )

        cls_tokens = self.cls_token.expand(B , -1, -1)#expand change variable but -1 keep the dimension unchanged 
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed #we add each vector with their dimensions 
        x = self.transformer_layers(x)
        x = x[:,0]#takes only the CLS so from [64,17,16] to [64,16]
        x = self.mlp_head(x) # from [64,16] to [64,10]
        return x
"""Conv2d weights
Linear weights
Attention WQ
Attention WK
Attention WV
MLP weights
        +
Position embeddings
        ↓
all are trainable parameters /*
"""


'Conv2d weights\nLinear weights\nAttention WQ\nAttention WK\nAttention WV\nMLP weights\n        +\nPosition embeddings\n        ↓\nall are trainable parameters /*\n'

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VisionTransformer().to(device)
optimizer = Adam(model.parameters(),lr=0.01)
criterion = nn.CrossEntropyLoss()

In [12]:
for epoch in range(5):
    model.train()
    total_loss = 0
    correct_epoch = 0
    total_epoch = 0
    print(f"\nEpoch {epoch+1}")

    for batch_idx, (images, labels) in enumerate(train_data):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss+=loss.item()
        preds = outputs.argmax(dim=1) #gets the correct label to compare 

        correct = (preds == labels).sum().item()
        accuracy = 100.0 * correct / labels.size(0)

        correct_epoch += correct
        total_epoch += labels.size(0)

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx+1:3d}: Loss = {loss.item():.4f}, Accuracy = {accuracy:.2f}%") #update every 100 batch

    epoch_acc = 100.0 * correct_epoch / total_epoch
    print(f"==> Epoch {epoch+1} Summary: Total Loss = {total_loss:.4f}, Accuracy = {epoch_acc:.2f}%")


Epoch 1
  Batch   1: Loss = 2.3804, Accuracy = 10.94%
  Batch 101: Loss = 1.4130, Accuracy = 48.44%
  Batch 201: Loss = 0.9185, Accuracy = 65.62%
  Batch 301: Loss = 0.8309, Accuracy = 75.00%
  Batch 401: Loss = 0.7903, Accuracy = 75.00%
  Batch 501: Loss = 0.7734, Accuracy = 73.44%
  Batch 601: Loss = 0.4117, Accuracy = 85.94%
  Batch 701: Loss = 0.5298, Accuracy = 81.25%
  Batch 801: Loss = 0.6153, Accuracy = 73.44%
  Batch 901: Loss = 0.3377, Accuracy = 90.62%
==> Epoch 1 Summary: Total Loss = 770.6031, Accuracy = 71.55%

Epoch 2
  Batch   1: Loss = 0.6280, Accuracy = 82.81%
  Batch 101: Loss = 0.4611, Accuracy = 87.50%
  Batch 201: Loss = 0.2785, Accuracy = 92.19%
  Batch 301: Loss = 0.5804, Accuracy = 82.81%
  Batch 401: Loss = 0.3335, Accuracy = 87.50%
  Batch 501: Loss = 0.2602, Accuracy = 89.06%
  Batch 601: Loss = 0.3418, Accuracy = 90.62%
  Batch 701: Loss = 0.5096, Accuracy = 81.25%
  Batch 801: Loss = 0.5757, Accuracy = 84.38%
  Batch 901: Loss = 0.7456, Accuracy = 79.69%
